In [8]:
import os
import cv2
from tqdm import tqdm

In [4]:
# --- STEP 3: FRAME EXTRACTION FUNCTIONS ---
def extract_frames(video_path, output_folder, frame_interval, image_format):
    """
    Extracts frames from a single video based on a specified interval.
    """
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error: Could not open video file: {video_path}")
            return

        video_name = os.path.splitext(os.path.basename(video_path))[0]
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        saved_frame_count = 0
        current_frame_index = 0

        for _ in tqdm(range(frame_count), desc=f"Extracting {video_name}", unit="frame", leave=False):
            ret, frame = cap.read()
            if not ret:
                break

            # Check if the current frame is one we should save
            if current_frame_index % frame_interval == 0:
                # Format the frame number with leading zeros for proper sorting
                frame_filename = f"{video_name}_frame_{saved_frame_count:05d}.{image_format}"
                output_path = os.path.join(output_folder, frame_filename)
                cv2.imwrite(output_path, frame)
                saved_frame_count += 1

            current_frame_index += 1

        cap.release()
        print(f"SUCCESS: Extracted {saved_frame_count} frames to: {output_folder}")

    except Exception as e:
        print(f"\nAn error occurred during frame extraction for {video_path}: {e}")

In [5]:
def process_directory_for_frame_extraction(input_dir, output_dir, frame_interval, image_format):
    """
    Recursively finds all videos in input_dir and extracts their frames.
    """
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    print(f"\nStarting to search for videos in '{input_dir}'...")
    for root, _, files in os.walk(input_dir):
        for filename in files:
            if filename.lower().endswith(video_extensions):
                video_path = os.path.join(root, filename)

                # Create a specific subfolder for each video's frames to keep them organized
                video_name_no_ext = os.path.splitext(filename)[0]
                relative_path = os.path.relpath(root, input_dir)
                frame_output_folder = os.path.join(output_dir, relative_path, video_name_no_ext)

                os.makedirs(frame_output_folder, exist_ok=True)

                extract_frames(video_path, frame_output_folder, frame_interval, image_format)

In [6]:
# --- CONFIGURATION ---
input_video_dir = ''
output_frame_dir = ''

# --- ADJUSTABLE PARAMETERS ---
# Save one frame every 'N' frames. 1 = every frame, 5 = every 5th frame, etc.
FRAME_INTERVAL = 1
# Output image format. 'jpg' is smaller, 'png' is higher quality.
IMAGE_FORMAT = 'png'
# -------------------

In [ ]:
print("\n===== STARTING FRAME EXTRACTION PROCESS =====")
if os.path.isdir(input_video_dir):
    process_directory_for_frame_extraction(
        input_dir=input_video_dir,
        output_dir=output_frame_dir,
        frame_interval=FRAME_INTERVAL,
        image_format=IMAGE_FORMAT
    )
    print("\n===== FRAME EXTRACTION COMPLETE =====")
else:
    print(f"Error: Input directory not found at '{input_video_dir}'. Please check the path.")


===== STARTING FRAME EXTRACTION PROCESS =====
Error: Input directory not found at ''. Please check the path.
